In [1]:
# Cell 1.1: data audit of the twelve AfriSenti languages, CPU only. kin runs first and must reproduce e501 before the other eleven run. Version 2: a wrapping pair of straight quotes is removed (the amh test file wraps every tweet), and the normalized tweets are saved for the later cells.
import re
from pathlib import Path
import pandas as pd
import pyarrow, datasets, huggingface_hub
from datasets import load_dataset
from huggingface_hub import HfApi

datasets.disable_progress_bars() # no progress bars while the splits load
huggingface_hub.utils.disable_progress_bars() # nor while the files download, so the printed output stays short

ROOT = Path.cwd().parent # this notebook sits in notebooks/ of the repository
assert (ROOT / "data").is_dir(), f"unexpected project folder: {ROOT}" # stop if the notebook is somewhere else
AUDIT_PATH = ROOT / "data" / "afrisenti_audit.csv" # the file that gets one row per language
NORM_PATH = ROOT / "data" / "afrisenti_normalized.parquet" # every tweet with its normalized text, for the later cells and the grid script; ignored by git

REPO = "shmuhammad/AfriSenti-twitter-sentiment" # the dataset on the Hugging Face Hub
LANGS = ["kin", "swa", "tso", "arq", "por", "twi", "pcm", "ary", "amh", "yor", "ibo", "hau"] # kin first, then the order the grid will run
SPLITS = ["train", "validation", "test"] # the official splits, no resampling
SHORT = {"train": "train", "validation": "val", "test": "test"} # short split names for the column labels
LABEL_NAMES = ["positive", "neutral", "negative"] # the ClassLabel names for ids 0, 1, 2

SIZES = {"amh": (5984, 1497, 1999), "arq": (1651, 414, 958), "ary": (5583, 494, 2961), "hau": (14172, 2677, 5303),
         "ibo": (10192, 1841, 3682), "kin": (3302, 827, 1026), "pcm": (5121, 1281, 4154), "por": (3063, 767, 3662),
         "swa": (1810, 453, 748), "tso": (804, 203, 254), "twi": (3481, 388, 949), "yor": (8522, 2090, 4515)} # train, validation, test rows measured on 26 September

EXPECTED_KIN = {"mention_runs": 1621, "links": 1607, "text_changed": 2950,
                "val_in_train": 32, "val_in_train_difflabel": 7, "test_in_train": 29, "test_in_train_difflabel": 7,
                "test_in_val": 8, "train_removed": 61, "train_dup_rows": 86, "train_dup_texts": 43, "train_dup_conflicts": 9,
                "train_final": 3241, "train_positive": 883, "train_neutral": 1237, "train_negative": 1121,
                "val_positive": 225, "val_neutral": 315, "val_negative": 287,
                "test_positive": 278, "test_neutral": 393, "test_negative": 355} # printed by cell 1.1 of my e501 notebook, version 11

APOSTROPHES = {"straight": "\u0027", # ' the only apostrophe the e301 rule removes
               "curly": "\u2019", # ’ as in n’umuryango
               "other": "\u2018\u0060\u00b4\u02bc"} # ‘ ` ´ ʼ look-alikes, such as the backtick in y`isahu (a kin tweet printed in my e301 cell 2.1)

WRAPPED = re.compile(r"^'(.*)'$", re.DOTALL) # a tweet whose first and last characters are straight quotes
MENTION_RUN = re.compile(r"(@user\b)(?:\s*@user\b)+", re.IGNORECASE) # a first @user followed by one or more @user
URL = re.compile(r"https?://\S+") # a link, up to the next space
SPACES = re.compile(r"\s+") # any run of whitespace

def normalise(text):
    text = WRAPPED.sub(r"\1", text.strip()) # drop the wrapping pair of quotes, a file artifact of the amh test split; new in this version
    text = MENTION_RUN.sub(r"\1", text) # keep the first mention, drop the ones that follow it
    text = URL.sub("http", text) # every link becomes the token http
    return SPACES.sub(" ", text).strip() # single spaces, nothing at the ends

SHA = HfApi().dataset_info(REPO, revision="refs/convert/parquet").sha # the commit the parquet branch points to today
assert isinstance(SHA, str) and len(SHA) == 40, f"unexpected commit id: {SHA}" # a full git commit hash
VERSIONS = {"parquet_commit": SHA, "pandas": pd.__version__, "pyarrow": pyarrow.__version__, "datasets": datasets.__version__,
            "huggingface_hub": huggingface_hub.__version__} # the data version and library versions, saved with the counts
print(VERSIONS)

def audit(lang):
    base = f"hf://datasets/{REPO}@{SHA}/{lang}" # this language's parquet files, pinned to that commit
    ds = load_dataset("parquet", data_files={s: f"{base}/{s}/*.parquet" for s in SPLITS}) # the three splits
    frames = [] # one DataFrame per split
    for s in SPLITS:
        feat = ds[s].features["label"] # the type of the label column
        assert getattr(feat, "names", None) == LABEL_NAMES, f"{lang} {s}: label feature is {feat}" # a ClassLabel with the same three names in the same order
        frames.append(ds[s].to_pandas().assign(split=s)) # the split as a DataFrame, its name in a new column
    data = pd.concat(frames, ignore_index=True) # all tweets of the language, splits in the released order
    sizes = tuple(int((data["split"] == s).sum()) for s in SPLITS) # rows per split as loaded
    assert sizes == SIZES[lang], f"{lang}: loaded {sizes}, measured {SIZES[lang]} on 26 September" # the same files as the source check
    assert data["label"].isin([0, 1, 2]).all(), f"{lang}: a label id outside 0, 1, 2" # no missing or extra label
    assert data["tweet"].map(lambda t: isinstance(t, str)).all(), f"{lang}: a tweet that is not text" # no missing tweet
    data["label_name"] = data["label"].map(dict(enumerate(LABEL_NAMES))) # the label as a word
    data["text"] = data["tweet"].map(normalise) # the e501 normalization on every tweet, plus the quote rule
    row = {"lang": lang, "train_raw": sizes[0], "val": sizes[1], "test": sizes[2]} # this language's line in the table

    row["mention_runs"] = int(data["tweet"].map(lambda t: bool(MENTION_RUN.search(t))).sum()) # tweets where a run of mentions was collapsed
    row["links"] = int(data["tweet"].map(lambda t: bool(URL.search(t))).sum()) # tweets where a link was replaced
    row["text_changed"] = int((data["tweet"] != data["text"]).sum()) # tweets the normalization changed in any way, spaces included
    row["empty_after_norm"] = int((data["text"] == "").sum()) # tweets with nothing left after normalization
    for s in SPLITS:
        row[f"wrapped_{SHORT[s]}"] = int(data.loc[data["split"] == s, "tweet"].map(lambda t: bool(WRAPPED.match(t.strip()))).sum()) # tweets the quote rule unwrapped

    train_mask = data["split"] == "train" # the training rows as released
    train_texts = set(data.loc[train_mask, "text"]) # every normalized training text
    train_pairs = set(zip(data.loc[train_mask, "text"], data.loc[train_mask, "label"])) # every (text, label) pair in train
    for s in ["validation", "test"]:
        part = data[data["split"] == s] # one evaluation split
        dup = part[part["text"].isin(train_texts)] # its tweets whose normalized text is also in train
        n_same = sum((t, l) in train_pairs for t, l in zip(dup["text"], dup["label"])) # copies whose label also goes with that text in train
        row[f"{SHORT[s]}_in_train"] = len(dup) # copies in train
        row[f"{SHORT[s]}_in_train_difflabel"] = len(dup) - n_same # copies whose label never goes with that text in train
    val_texts = set(data.loc[data["split"] == "validation", "text"]) # every normalized validation text
    row["test_in_val"] = int(data.loc[data["split"] == "test", "text"].isin(val_texts).sum()) # test tweets also in validation, both kept as published

    eval_texts = set(data.loc[~train_mask, "text"]) # every validation and test text
    drop = train_mask & data["text"].isin(eval_texts) # training rows that copy an evaluation tweet
    data.loc[drop, "split"] = "train_removed" # set aside, as in e501
    train_final = data[data["split"] == "train"] # the training set the models will see
    dup_train = train_final[train_final.duplicated("text", keep=False)] # every training row whose text appears more than once
    row["train_removed"] = int(drop.sum()) # rows removed from train
    row["train_dup_rows"] = len(dup_train) # rows that share their text with another training row
    row["train_dup_texts"] = int(dup_train["text"].nunique()) # how many distinct texts those rows hold
    row["train_dup_conflicts"] = int((dup_train.groupby("text")["label"].nunique() > 1).sum()) # texts that carry two or more labels, kept
    row["train_final"] = len(train_final) # the final training size

    for s in SPLITS:
        counts = data.loc[data["split"] == s, "label"].value_counts() # tweets per label id; train is now the final training set
        for i, name in enumerate(LABEL_NAMES):
            row[f"{SHORT[s]}_{name}"] = int(counts.get(i, 0)) # for example train_positive
    for name, chars in APOSTROPHES.items():
        for s in SPLITS:
            texts = data.loc[data["split"] == s, "text"] # normalized text, the input the retyping rules will see
            row[f"{name}_{SHORT[s]}"] = int(texts.map(lambda t: any(c in t for c in chars)).sum()) # tweets holding at least one of these characters
    return row, data.assign(lang=lang)[["lang", "split", "label", "label_name", "tweet", "text"]] # the counts, and the tweets with their normalized text

rows, parts = [], [] # one dictionary of counts per language, and one DataFrame of tweets per language
for lang in LANGS:
    row, part = audit(lang) # load, normalize and count one language
    rows.append(row); parts.append(part)
    print(f"{lang}: {row['train_final']} training tweets after the copies are removed, {row['wrapped_train'] + row['wrapped_val'] + row['wrapped_test']} tweets unwrapped")
    if lang == "kin":
        diff = {k: (rows[-1][k], v) for k, v in EXPECTED_KIN.items() if rows[-1][k] != v} # every count that differs, as (found, expected)
        assert not diff, f"kin does not reproduce e501, so the audit stops here: {diff}"
        print(f"kin reproduces all {len(EXPECTED_KIN)} e501 counts, so the other eleven languages follow")

normalized = pd.concat(parts, ignore_index=True) # all 106,828 tweets, the removed training rows marked train_removed
normalized.to_parquet(NORM_PATH, index=False) # saved for the later cells
back = pd.read_parquet(NORM_PATH, columns=["lang", "split"]) # read back
assert len(back) == len(normalized) == sum(sum(SIZES[l]) for l in LANGS), "the normalized file does not read back"
print(f"Saved {NORM_PATH.relative_to(ROOT.parent)}: {len(back)} rows; rows per split {back['split'].value_counts().to_dict()}")
audit_df = pd.DataFrame(rows).assign(**VERSIONS) # one row per language, the commit and versions repeated on every row
audit_df.to_csv(AUDIT_PATH, index=False) # written the moment the counts exist
check = pd.read_csv(AUDIT_PATH) # read the file back
assert check.shape == audit_df.shape and check["lang"].tolist() == LANGS, "the saved file does not read back"
print(f"\nSaved {AUDIT_PATH.relative_to(ROOT.parent)}: {check.shape[0]} rows, {check.shape[1]} columns\n")
table = audit_df.drop(columns=list(VERSIONS)).set_index("lang").T # one row per measure, one column per language
table["all"] = table.sum(axis=1) # the twelve languages together
print(table.to_string())

{'parquet_commit': '676cfb15a6761c643d860dcb53bac976e5db2dfc', 'pandas': '3.0.3', 'pyarrow': '24.0.0', 'datasets': '5.0.0', 'huggingface_hub': '0.36.2'}
kin: 3241 training tweets after the copies are removed, 0 tweets unwrapped
kin reproduces all 22 e501 counts, so the other eleven languages follow
swa: 1806 training tweets after the copies are removed, 0 tweets unwrapped
tso: 804 training tweets after the copies are removed, 0 tweets unwrapped
arq: 1649 training tweets after the copies are removed, 0 tweets unwrapped
por: 3030 training tweets after the copies are removed, 0 tweets unwrapped
twi: 3212 training tweets after the copies are removed, 0 tweets unwrapped
pcm: 3875 training tweets after the copies are removed, 0 tweets unwrapped
ary: 5581 training tweets after the copies are removed, 0 tweets unwrapped
amh: 5984 training tweets after the copies are removed, 1999 tweets unwrapped
yor: 8285 training tweets after the copies are removed, 0 tweets unwrapped
ibo: 10154 training twe

**Note on Cell 1.1.** The audit loaded the parquet conversion of AfriSenti at commit 676cfb15a6761c643d860dcb53bac976e5db2dfc, with pandas 3.0.3, pyarrow 24.0.0, datasets 5.0.0 and huggingface_hub 0.36.2. Every split of every language carries the same ClassLabel (0 positive, 1 neutral, 2 negative), every label id is 0, 1 or 2, the split sizes are the ones measured on 26 September (63,685 training, 12,932 validation and 30,211 test tweets), and no tweet is empty after normalization. This second version adds one rule to the e501 normalization, logged as a change: a tweet whose first and last characters are straight quotes loses that pair. The first run of this cell had shown a straight apostrophe in all 1,999 amh test tweets and in none of the amh training or validation tweets, which is a file artifact, not language. The rule touched exactly those 1,999 tweets and nothing else in the twelve languages. The cell also writes every tweet with its normalized text to data/afrisenti_normalized.parquet (106,828 rows, the removed training rows marked train_removed), which the later cells and the grid script read, so every later number rests on the same texts counted here.

Kinyarwanda reproduces all 22 counts printed by cell 1.1 of my e501 notebook: 1,621 mention runs, 1,607 links, 2,950 texts changed, 32 validation and 29 test copies in train with 7 and 7 different labels, 8 test tweets in validation, 61 training rows removed, 86 duplicated rows over 43 texts with 9 label conflicts, and a final training set of 3,241 tweets (883 positive, 1,237 neutral, 1,121 negative) against 827 validation and 1,026 test tweets. So the normalization and the removal are the same operations as in e501, and the other eleven languages went through them unchanged.

Normalization. Mention runs were collapsed in 10,587 tweets and links replaced in 10,174; with the quote rule, 24,465 of the 106,828 tweets changed in some way (3,844 of the 9,480 amh tweets). swa, twi and pcm have no mention runs, and swa, arq, twi, pcm and amh have no links. twi and pcm did not change at all, so their files were released already clean; swa changed in 548 tweets through whitespace alone.

Copies between splits. 894 validation and 1,151 test tweets have a copy in train, and 71 and 163 of them carry a label that never goes with that text in train. 288 test tweets are also in validation (kin 8, twi 24, pcm 256); both splits stay as published. The removal took 2,107 training rows, 3.3 percent of the released training data, from none in tso and amh to 1,246 in pcm, 24.3 percent of its 5,121 rows. In pcm 926 of the 4,154 test tweets have a copy in train (22.3 percent), in twi 195 of 949 (20.5 percent). The twelve training sets hold 61,578 tweets after removal.

Duplicates inside train, kept and counted: 1,817 rows over 889 texts, and 135 of those texts (15.2 percent) carry two or more labels. twi has 752 rows over 376 texts with 87 conflicts, pcm 740 over 351 with 17, por 160 over 80 with 19, kin 86 over 43 with 9. tso, yor, ibo and hau have none.

Label balance. The balance moves between splits in several languages. pcm has 55 neutral tweets in its final training set (1.4 percent of 3,875) and 21 in validation, but 431 in test (10.4 percent). amh test is 66.9 percent negative (1,337 of 1,999) where its training set is 25.9 percent negative. por test is 65.0 percent neutral (2,379 of 3,662) where its training set is 52.1 percent neutral. These go into Table 1 and into Limitations.

Apostrophes, counted on the normalized text. Kinyarwanda uses both forms in every split: the straight one in 665, 183 and 236 tweets (train, validation, test), the curly one in 189, 53 and 67, and a look-alike in 7, 0 and 2. In the test set that is 23.0 percent of tweets with a straight apostrophe and 6.5 percent with a curly one. swa, arq and pcm have no apostrophe of any form in any split, and twi has two look-alikes in train and nothing else. With the amh quotes gone, four languages still show a split mismatch that cannot come from language use: yor, ibo and hau have a straight apostrophe in 23.4, 12.7 and 8.8 percent of their training tweets and in no test tweet at all, and ary has it in 32.6 percent of training tweets and 1.5 percent of test tweets. Nothing can put those apostrophes back, so the test sets stay as published, and this goes into Limitations as a property of the released files: for these four languages every elided form in the test set is a word the training set never showed, which the out-of-vocabulary rate will reflect.

In [2]:
# Cell 1.2: the retyped Kinyarwanda test set. The e301 texting rules run once over the 1,026 normalized test tweets, in the released order, with seed 42, and the result is saved before any model sees it. Version 2: the apostrophe rule covers the curly apostrophe as well.
import random, re
from pathlib import Path
import pandas as pd
import datasets
from datasets import load_dataset

datasets.disable_progress_bars() # no progress bars while the split loads

ROOT = Path.cwd().parent # this notebook sits in notebooks/ of the repository
assert (ROOT / "data").is_dir(), f"unexpected project folder: {ROOT}" # stop if the notebook is somewhere else
AUDIT_PATH = ROOT / "data" / "afrisenti_audit.csv" # written by Cell 1.1
OUT_PATH = ROOT / "data" / "kin_test_retyped.csv" # the file every kin pipeline will score

SEED = 42 # the seed of e301 and e501
REPO = "shmuhammad/AfriSenti-twitter-sentiment" # the dataset on the Hugging Face Hub
LABEL_NAMES = ["positive", "neutral", "negative"] # the ClassLabel names for ids 0, 1, 2

DROP = "\u0027\u2019" # decision 1: the characters the apostrophe rule removes, the straight apostrophe ' as in e301 and the curly one ’ (a logged change: the retyped set carries no apostrophe at all)
PROTECTED = {"@user", "http"} # decision 2: placeholders the rules leave alone; set() lets them change as in e301

audit = pd.read_csv(AUDIT_PATH) # the audit, one row per language
commits = audit["parquet_commit"].unique() # the parquet commit Cell 1.1 loaded
assert len(commits) == 1, f"more than one commit in the audit file: {commits}"
SHA = commits[0]
KIN = audit.set_index("lang").loc["kin"] # the kin row of the audit, for the checks below

MENTION_RUN = re.compile(r"(@user\b)(?:\s*@user\b)+", re.IGNORECASE) # a first @user followed by one or more @user
URL = re.compile(r"https?://\S+") # a link, up to the next space
SPACES = re.compile(r"\s+") # any run of whitespace

def normalise(text):
    text = MENTION_RUN.sub(r"\1", text) # keep the first mention, drop the ones that follow it
    text = URL.sub("http", text) # every link becomes the token http
    return SPACES.sub(" ", text).strip() # single spaces, nothing at the ends

noise_rng = random.Random(SEED) # one random stream for the whole test set, as in e301
ABBREV = {"kandi": "kdi", "cyangwa": "cg", # seen in the corpus
          "ariko": "ark", "kubera": "kbr", "gusa": "gsa", "neza": "nza"}
VOWELS = "aeiou"

def stretch(w): return w + w[-1] * 2 if w[-1].lower() in VOWELS else w # cyane -> cyaneee
def drop_vowel(w):
    inner = [i for i, ch in enumerate(w[1:-1], 1) if ch.lower() in VOWELS] # positions of the inner vowels
    return w if len(w) < 6 or not inner else (lambda i: w[:i] + w[i+1:])(noise_rng.choice(inner)) # one inner vowel dropped, words under six letters untouched
def swap_adjacent(w):
    if len(w) < 5: return w # words under five letters untouched
    i = noise_rng.randrange(1, len(w) - 2); return w[:i] + w[i+1] + w[i] + w[i+2:] # two inner neighbors swapped

def drop_apostrophes(text):
    for c in DROP: text = text.replace(c, "") # the first e301 rule, now for both apostrophes
    return text

def add_noise(text):
    base = drop_apostrophes(text) # apostrophes dropped before the word rules run
    for _ in range(10): # up to ten passes, until something changes
        out, changed = [], base != text
        for w in base.split():
            lw = w.lower()
            if lw in PROTECTED: out.append(w); continue # a placeholder stays as it is
            if lw in ABBREV and noise_rng.random() < 0.8:
                out.append(ABBREV[lw]); changed = True; continue # the texting abbreviation, eight times in ten
            r = noise_rng.random()
            if r < 0.10: nw = stretch(w)
            elif r < 0.20: nw = drop_vowel(w)
            elif r < 0.27: nw = swap_adjacent(w)
            else: nw = w
            changed |= nw != w; out.append(nw)
        if changed: return " ".join(out), False # retyped by the rules
    free = [i for i, w in enumerate(out) if w.lower() not in PROTECTED] # the words the fallback may change
    if not free: return " ".join(out), False # nothing but placeholders, left as it is
    out[free[-1]] = stretch(out[free[-1]] + "e") # fallback: force one change on the last ordinary word, as in e301
    return " ".join(out), True

ds = load_dataset("parquet", data_files={"test": f"hf://datasets/{REPO}@{SHA}/kin/test/*.parquet"})["test"] # the kin test split at the audited commit
assert getattr(ds.features["label"], "names", None) == LABEL_NAMES, f"label feature is {ds.features['label']}" # the same ClassLabel as in the audit
test = ds.to_pandas() # the released order
assert len(test) == KIN["test"], f"{len(test)} test tweets, the audit counted {KIN['test']}"
test["text"] = test["tweet"].map(normalise) # the e501 normalization, as in Cell 1.1
for i, name in enumerate(LABEL_NAMES):
    assert int((test["label"] == i).sum()) == KIN[f"test_{name}"], f"{name} count differs from the audit" # the label balance the audit saved
test["label_name"] = test["label"].map(dict(enumerate(LABEL_NAMES)))

retyped = test["text"].map(add_noise) # (retyped text, whether the fallback was needed), in the released order
test["retyped"] = retyped.map(lambda r: r[0])
test["forced"] = retyped.map(lambda r: r[1])
test["changed"] = test["retyped"] != test["text"] # the column that says whether each tweet changed
test["apostrophe_dropped"] = test["text"].map(lambda t: any(c in t for c in DROP)) # tweets the first rule touched
out = test.reset_index().rename(columns={"index": "idx"})[["idx", "label", "label_name", "tweet", "text", "retyped", "changed", "apostrophe_dropped", "forced"]]
out.to_csv(OUT_PATH, index=False) # saved before any model sees it
check = pd.read_csv(OUT_PATH) # read back
assert check.shape == out.shape and check["changed"].sum() == out["changed"].sum(), "the saved file does not read back"

n = len(out)
base_words = test["text"].map(lambda t: drop_apostrophes(t).split()) # the words each tweet had when the word rules ran
new_words = test["retyped"].map(lambda t: t.split()) # the words of each retyped tweet
assert (base_words.map(len) == new_words.map(len)).all(), "a word rule changed the number of words" # every word rule keeps one word as one word
n_words = int(base_words.map(len).sum())
n_words_changed = int(sum(sum(a != b for a, b in zip(ws, ns)) for ws, ns in zip(base_words, new_words))) # words the word rules changed, apostrophes aside
n_user = int(base_words.map(lambda ws: sum(w.lower() == "@user" for w in ws)).sum()) # @user placeholders in the test set
n_user_changed = int(sum(sum(a.lower() == "@user" and a != b for a, b in zip(ws, ns)) for ws, ns in zip(base_words, new_words)))
curly_left = int(test["retyped"].map(lambda t: "\u2019" in t).sum()) # tweets that still hold a curly apostrophe after retyping
print(f"Saved {OUT_PATH.relative_to(ROOT.parent)}: {check.shape[0]} rows, {check.shape[1]} columns")
print(f"tweets changed: {int(out['changed'].sum())} of {n}; the apostrophe rule touched {int(out['apostrophe_dropped'].sum())}; the fallback forced a change in {int(out['forced'].sum())}")
print(f"words changed by the word rules: {n_words_changed} of {n_words}; @user placeholders: {n_user}, changed {n_user_changed}; tweets that keep a curly apostrophe: {curly_left}")
print(f"label balance: {out['label_name'].value_counts().reindex(LABEL_NAMES).to_dict()}")
pd.set_option("display.max_colwidth", None)
print("\nThe first three changed tweets, clean and retyped:")
for r in out[out["changed"]].head(3).itertuples():
    print(f"  [{r.label_name}] {r.text}\n  -> {r.retyped}")

Saved final_project/data/kin_test_retyped.csv: 1026 rows, 9 columns
tweets changed: 1026 of 1026; the apostrophe rule touched 303; the fallback forced a change in 1
words changed by the word rules: 2679 of 14188; @user placeholders: 818, changed 0; tweets that keep a curly apostrophe: 0
label balance: {'positive': 278, 'neutral': 393, 'negative': 355}

The first three changed tweets, clean and retyped:
  [three example tweets were printed here; this public copy does not store tweets, so run the cell to see them]


**Note on Cell 1.2.** The e301 rules ran once over the 1,026 normalized Kinyarwanda test tweets, in the released order, with seed 42, and the retyped set is saved in data/kin_test_retyped.csv with the raw tweet, the clean text, the retyped text and a column that says whether each tweet changed. Every tweet changed, which the rules guarantee: when no rule fires in ten passes, the fallback changes the last ordinary word, and that happened once. The apostrophe rule touched 303 tweets, 29.5 percent of the test set: the 236 with a straight apostrophe and the 67 with a curly one, and no tweet had both. This is the one place where the rules differ from e301, logged as a change (decision 1): e301 dropped only the straight apostrophe, so a tweet typed with a curly one kept it, and the retyped set would then have carried two kinds of apostrophe behavior. No tweet keeps a curly apostrophe now. The word rules changed 2,679 of the 14,188 words (18.9 percent). The 818 @user placeholders and the http tokens were left alone (decision 2): they were inserted by the dataset's anonymization and are not something a person types, so the swap rule would only have produced forms such as @uesr that say nothing about Kinyarwanda. The label balance is the test set's, 278 positive, 393 neutral and 355 negative. In the three examples printed above, ntabwo, urukiko and rubyemeje were stretched, icyaha, Itegeko and Abafollowers had two inner letters swapped, uretse lost a vowel, and ry'umuryango, n'umugore and w'urugo lost their apostrophes.

In [3]:
# Cell 1.3: tokens per word, unknown tokens and the out-of-vocabulary rate for the twelve languages, CPU only. kin must reproduce the e501 tokenizer numbers before the table is saved.
import re
from pathlib import Path
import numpy as np
import pandas as pd
import nltk, transformers
from nltk.tokenize import TweetTokenizer
from transformers import AutoTokenizer
from huggingface_hub import HfApi

ROOT = Path.cwd().parent # this notebook sits in notebooks/ of the repository
assert (ROOT / "data").is_dir(), f"unexpected project folder: {ROOT}" # stop if the notebook is somewhere else
NORM_PATH = ROOT / "data" / "afrisenti_normalized.parquet" # every tweet with its normalized text, written by Cell 1.1
AUDIT_PATH = ROOT / "data" / "afrisenti_audit.csv" # written by Cell 1.1
RETYPED_PATH = ROOT / "data" / "kin_test_retyped.csv" # written by Cell 1.2
OUT_PATH = ROOT / "data" / "afrisenti_fertility.csv" # one row per language
TOKENS_PATH = ROOT / "data" / "kin_retyped_tokens.csv" # one row per kin test tweet, clean and retyped token counts

LANGS = ["kin", "swa", "tso", "arq", "por", "twi", "pcm", "ary", "amh", "yor", "ibo", "hau"] # the order of the audit
MODELS = {"afroxlmr": "Davlan/afro-xlmr-base", "labse": "sentence-transformers/LaBSE"} # the two vocabularies the pipelines use
MAX_LEN = 128 # the AfroXLMR truncation limit of e501
TOKEN_PATTERN = re.compile(r"\w+(?:['’‘\-–]\w+)*['’‘]?") # the e501 TF-IDF word pattern: apostrophes and hyphens stay inside a word
APOS = re.compile("['’‘]") # the three apostrophe forms the pattern keeps
PLACEHOLDERS = {"@user", "http"} # the tokens that are not language
EXPECTED_KIN = {"labse_tpw": 1.91, "labse_unk": 61, "labse_longest": 82,
                "afroxlmr_tpw": 2.53, "afroxlmr_unk": 119, "afroxlmr_longest": 98, "afroxlmr_over128_train": 0} # printed by cells 1.4 and 1.5 of my e501 notebook, version 11

audit = pd.read_csv(AUDIT_PATH).set_index("lang") # the audit counts, for the checks
data = pd.read_parquet(NORM_PATH, columns=["lang", "split", "text"]) # the normalized tweets
assert len(data) == 106828, f"{len(data)} rows in the normalized file" # the twelve languages, all splits
api = HfApi()
COMMITS = {name: api.model_info(repo).sha for name, repo in MODELS.items()} # the commit each tokenizer comes from today
toks = {name: AutoTokenizer.from_pretrained(repo, revision=COMMITS[name]) for name, repo in MODELS.items()} # the tokenizers, pinned to those commits
tweet_tok = TweetTokenizer() # the e101 alternative word counter
VERSIONS = {f"{name}_commit": sha for name, sha in COMMITS.items()} | {"transformers": transformers.__version__, "nltk": nltk.__version__}
print({name: f"{type(t).__name__}, vocabulary {t.vocab_size}, unk id {t.unk_token_id}, commit {COMMITS[name][:12]}" for name, t in toks.items()})

def content_only(text):
    return " ".join(w for w in text.split() if w.lower() not in PLACEHOLDERS and re.search(r"\w", w)) # the words that carry language: no placeholders, no emoji or punctuation on their own

def words(texts):
    return [w.lower() for t in texts for w in TOKEN_PATTERN.findall(t)] # every word occurrence under the TF-IDF pattern, lowercased

def token_stats(name, texts):
    tok = toks[name]
    ids = tok(list(texts), add_special_tokens=False)["input_ids"] # subword ids without the special tokens, as in e501
    lens = np.array([len(i) for i in ids]) # tokens per tweet
    return {"tokens": int(lens.sum()), "unk": int(sum(i.count(tok.unk_token_id) for i in ids)), # unknown tokens
            "longest": int(lens.max()) + 2, "over128": int(((lens + 2) > MAX_LEN).sum())} # the longest tweet with its two specials, and the tweets the limit would cut

rows = []
for lang in LANGS:
    part = data[data["lang"] == lang]
    train = part.loc[part["split"] == "train", "text"].tolist() # the final training set, copies of evaluation tweets removed
    test = part.loc[part["split"] == "test", "text"].tolist() # the test set as published
    assert len(train) == audit.loc[lang, "train_final"] and len(test) == audit.loc[lang, "test"], f"{lang}: sizes differ from the audit"
    row = {"lang": lang, "train_final": len(train), "test": len(test)}
    row["words_ws"] = sum(len(t.split()) for t in train) # whitespace words, the e501 denominator
    row["words_tweettok"] = sum(len(tweet_tok.tokenize(t)) for t in train) # the same tweets counted by NLTK's TweetTokenizer
    row["words_pattern"] = sum(len(TOKEN_PATTERN.findall(t)) for t in train) # and by the TF-IDF word pattern
    content = [content_only(t) for t in train] # the tweets without placeholders, emoji and punctuation
    row["words_content"] = sum(len(t.split()) for t in content)
    for name in MODELS:
        s = token_stats(name, train)
        row[f"{name}_tpw"] = s["tokens"] / row["words_ws"] # the primary measure: subword tokens per whitespace word on the training tweets
        row[f"{name}_tpw_tweettok"] = s["tokens"] / row["words_tweettok"] # the same tokens over the TweetTokenizer count
        row[f"{name}_tpw_pattern"] = s["tokens"] / row["words_pattern"] # and over the pattern count
        row[f"{name}_tpw_content"] = token_stats(name, content)["tokens"] / row["words_content"] # tokens and words both counted without placeholders, emoji and punctuation
        row[f"{name}_unk"] = s["unk"] # unknown tokens in the training tweets
        row[f"{name}_longest"] = s["longest"] # the longest training tweet in tokens, specials included
        row[f"{name}_over128_train"] = s["over128"] # training tweets the 128 limit would cut
        row[f"{name}_over128_test"] = token_stats(name, test)["over128"] # test tweets it would cut
    train_vocab = set(words(train)) # every distinct training word
    test_words = words(test) # every test word occurrence
    row["train_vocab"] = len(train_vocab)
    row["test_word_occurrences"] = len(test_words)
    row["oov_rate"] = sum(w not in train_vocab for w in test_words) / len(test_words) # the e201 measure: share of test word occurrences never seen in training
    train_vocab_noapos = {APOS.sub("", w) for w in train_vocab} # the same words with the apostrophes removed
    test_words_noapos = [APOS.sub("", w) for w in test_words]
    row["oov_rate_noapos"] = sum(w not in train_vocab_noapos for w in test_words_noapos) / len(test_words_noapos) # the same rate when apostrophes are ignored on both sides
    rows.append(row)
    print(f"{lang}: {row['afroxlmr_tpw']:.2f} and {row['labse_tpw']:.2f} tokens per word (AfroXLMR, LaBSE), OOV {row['oov_rate']:.1%}")
    if lang == "kin":
        found = {k: (float(f"{row[k]:.2f}") if k.endswith("_tpw") else row[k]) for k in EXPECTED_KIN} # tokens per word rounded as e501 printed them
        diff = {k: (found[k], v) for k, v in EXPECTED_KIN.items() if found[k] != v}
        assert not diff, f"kin does not reproduce the e501 tokenizer numbers, so the cell stops here: {diff}"
        print(f"kin reproduces all {len(EXPECTED_KIN)} e501 tokenizer numbers, so the other eleven languages follow")

fert = pd.DataFrame(rows).assign(**VERSIONS) # one row per language, the commits and versions repeated on every row
fert.to_csv(OUT_PATH, index=False)
check = pd.read_csv(OUT_PATH)
assert check.shape == fert.shape and check["lang"].tolist() == LANGS, "the saved file does not read back"
print(f"\nSaved {OUT_PATH.relative_to(ROOT.parent)}: {check.shape[0]} rows, {check.shape[1]} columns")

retyped = pd.read_csv(RETYPED_PATH) # the clean and retyped kin test tweets, in the released order
assert len(retyped) == audit.loc["kin", "test"] and retyped["idx"].tolist() == list(range(len(retyped))), "the retyped file is not the full test set in order"
tokens = retyped[["idx"]].copy()
for col in ["text", "retyped"]:
    tokens[f"words_{col}"] = retyped[col].map(lambda t: len(t.split())) # whitespace words per tweet
    for name in MODELS:
        ids = toks[name](retyped[col].tolist(), add_special_tokens=False)["input_ids"]
        tokens[f"{name}_{col}"] = [len(i) for i in ids] # subword tokens per tweet
        tokens[f"{name}_unk_{col}"] = [i.count(toks[name].unk_token_id) for i in ids] # unknown tokens per tweet
tokens.to_csv(TOKENS_PATH, index=False)
check2 = pd.read_csv(TOKENS_PATH)
assert check2.shape == tokens.shape, "the token file does not read back"
print(f"Saved {TOKENS_PATH.relative_to(ROOT.parent)}: {check2.shape[0]} rows, {check2.shape[1]} columns")
kin_train_vocab = set(words(data.loc[(data["lang"] == "kin") & (data["split"] == "train"), "text"]))
for col in ["text", "retyped"]:
    ws = tokens[f"words_{col}"].sum()
    ww = words(retyped[col])
    oov = sum(w not in kin_train_vocab for w in ww) / len(ww)
    print(f"kin test, {'clean' if col == 'text' else 'retyped'}: " + ", ".join(f"{name} {tokens[f'{name}_{col}'].sum() / ws:.3f} tokens per word with {tokens[f'{name}_unk_{col}'].sum()} unknown" for name in MODELS) + f"; OOV {oov:.1%}")

table = fert.drop(columns=list(VERSIONS)).set_index("lang").T # one row per measure, one column per language
print("\n" + table.apply(lambda r: r.map(lambda v: f"{v:.3f}") if ("tpw" in r.name or "rate" in r.name) else r.map(lambda v: f"{int(v)}"), axis=1).to_string()) # three decimals for the ratios, plain integers for the counts

{'afroxlmr': 'XLMRobertaTokenizerFast, vocabulary 250002, unk id 3, commit 25f27299c247', 'labse': 'BertTokenizerFast, vocabulary 501153, unk id 100, commit 836121a0533e'}
kin: 2.53 and 1.91 tokens per word (AfroXLMR, LaBSE), OOV 26.5%
kin reproduces all 7 e501 tokenizer numbers, so the other eleven languages follow
swa: 1.61 and 1.41 tokens per word (AfroXLMR, LaBSE), OOV 21.2%
tso: 2.02 and 1.82 tokens per word (AfroXLMR, LaBSE), OOV 28.9%
arq: 2.15 and 2.12 tokens per word (AfroXLMR, LaBSE), OOV 33.2%
por: 1.57 and 1.46 tokens per word (AfroXLMR, LaBSE), OOV 13.0%
twi: 1.87 and 1.53 tokens per word (AfroXLMR, LaBSE), OOV 17.4%
pcm: 1.23 and 1.10 tokens per word (AfroXLMR, LaBSE), OOV 6.8%
ary: 2.05 and 1.94 tokens per word (AfroXLMR, LaBSE), OOV 35.1%
amh: 2.33 and 2.34 tokens per word (AfroXLMR, LaBSE), OOV 35.0%
yor: 2.68 and 2.02 tokens per word (AfroXLMR, LaBSE), OOV 6.9%
ibo: 2.12 and 1.69 tokens per word (AfroXLMR, LaBSE), OOV 10.6%
hau: 1.85 and 1.59 tokens per word (AfroXLMR

**Note on Cell 1.3.** The two tokenizers came from the Hub at the commits recorded in the file (AfroXLMR at 25f27299c247, a 250,002-token XLM-R vocabulary; LaBSE at 836121a0533e, a 501,153-token WordPiece vocabulary), and the grid script loads the same commits. Kinyarwanda reproduces all seven e501 tokenizer numbers: 2.53 tokens per whitespace word, 119 unknown tokens, a longest tweet of 98 tokens and none over 128 under AfroXLMR; 1.91, 61 and 82 under LaBSE. So the measurement is the e501 measurement, applied unchanged to the other eleven languages on their final training sets.

Tokens per word. Under AfroXLMR the twelve languages run from 1.23 (pcm) to 2.69 (yor), with kin at 2.53 and amh at 2.33; arq, ibo, ary and tso sit between 2.0 and 2.2, twi and hau near 1.85, swa and por near 1.6. Under LaBSE every language costs less, from 1.10 (pcm) to 2.34 (amh), and the order changes: amh and arq are the most fragmented there, and kin is at 1.91. The gap between the two vocabularies is largest for yor (2.69 against 2.02), kin (2.53 against 1.91) and ibo (2.12 against 1.69), and close to zero for amh and arq, which use the Ethiopic and Arabic scripts. Unknown tokens are rare in both vocabularies: the most are 730 and 712 under AfroXLMR for yor and hau, the two largest training sets, and 382 under LaBSE for hau; swa has none.

Truncation. AfroXLMR cuts a tweet at 128 tokens, as in e501. That touches yor above all, 454 training tweets (5.5 percent) and 138 test tweets (3.1 percent), and a handful elsewhere (ibo 11, ary 6, hau 3 and twi 1 in train; por 1 in test). LaBSE's limit is 256 tokens (my e501 notebook printed max_seq_length 256) and the longest tweet anywhere is 225 tokens (ary), so LaBSE truncates nothing.

Word definitions (e101). NLTK's TweetTokenizer counts more words than whitespace, because it splits off punctuation, emoji and the curly apostrophe, so the ratio drops, for kin from 2.53 to 2.15; the TF-IDF pattern and the content-only count stay close to the whitespace count. The extremes do not move: pcm is the least fragmented under every definition, and yor and kin are the two most fragmented under every definition, though kin edges ahead of yor under the pattern count (2.56 against 2.55). The middle of the order moves by one or two places (hau and swa, arq and ary, tso and twi swap under the TweetTokenizer count). The statistics cell will give the rank correlation between the definitions; the primary measure stays the e501 one.

OOV. The share of test word occurrences never seen in training runs from 6.8 percent (pcm) to 35.1 percent (ary), with amh at 35.0, arq 33.2, tso 28.9 and kin 26.5; the four largest training sets (hau, yor, ibo, pcm) have the lowest rates, so training size has to be controlled for in the statistics. Ignoring apostrophes on both sides changes the rate by at most 0.7 points (kin 26.5 to 25.8, ibo 10.6 to 10.0, yor 6.9 to 6.5, hau 7.4 to 7.2), so the release's apostrophe stripping in yor, ibo, hau and ary inflates their OOV rates only slightly.

The retyped set. On the clean kin test set AfroXLMR spends 2.55 tokens per word and LaBSE 1.91; on the retyped set 2.62 and 2.02, a rise of 2.7 and 6.2 percent, with the unknown tokens unchanged (38 and 23). At the word level the retyping moves far more: the OOV rate goes from 26.5 to 37.8 percent. So the subword vocabularies absorb the retyping as a few more tokens, while a word-level model meets eleven points more unseen words; the retyping test will show which of the three pipelines pays for that. The per-tweet counts are in data/kin_retyped_tokens.csv for the paired bootstrap.

In [4]:
# Cell 1.4: the pretraining coverage class of each language, read from the papers and model cards, written to a file for the statistics cell. No computation here.
from pathlib import Path
import pandas as pd

ROOT = Path.cwd().parent # this notebook sits in notebooks/ of the repository
assert (ROOT / "data").is_dir(), f"unexpected project folder: {ROOT}" # stop if the notebook is somewhere else
OUT_PATH = ROOT / "data" / "coverage.csv"

ALABI = "Alabi et al. 2022, arXiv 2204.06487, Table 9 (Appendix A.1); model card huggingface.co/Davlan/afro-xlmr-base" # the twenty languages of the adaptive fine-tuning that made AfroXLMR
CONNEAU = "Conneau et al. 2020, arXiv 1911.02116, Table 6 (Appendix A)" # the 100 CC-100 languages XLM-R was pretrained on
FENG = "Feng et al. 2022, arXiv 2007.01852, Table 10 (Appendix C)" # the 109 languages of LaBSE

rows = [ # lang, in the AfroXLMR adaptation, in CC-100, in the LaBSE list, note; the class columns are derived below
    ("kin", True, False, True, ""),
    ("swa", True, True, True, ""),
    ("tso", False, False, False, ""),
    ("arq", True, True, True, "covered as Standard Arabic (ar) in all three lists; Algerian Arabic is named in none"),
    ("por", False, True, True, "covered as Portuguese (pt); the tweets are Mozambican Portuguese"),
    ("twi", False, False, False, ""),
    ("pcm", True, False, False, "Naija (pcm) in the AfroXLMR adaptation only"),
    ("ary", True, True, True, "covered as Standard Arabic (ar) in all three lists; Moroccan Arabic is named in none"),
    ("amh", True, True, True, ""),
    ("yor", True, False, True, ""),
    ("ibo", True, False, True, ""),
    ("hau", True, True, True, ""),
]
cov = pd.DataFrame(rows, columns=["lang", "in_afroxlmr_adaptation", "in_xlmr_cc100", "in_labse_list", "note"])
cov["afroxlmr_class"] = ["adapted" if a else ("xlmr_only" if x else "neither") for a, x in zip(cov["in_afroxlmr_adaptation"], cov["in_xlmr_cc100"])] # adapted beats xlmr_only beats neither
cov["labse_class"] = ["listed" if l else "not_listed" for l in cov["in_labse_list"]]
cov["source_afroxlmr"] = ALABI
cov["source_xlmr"] = CONNEAU
cov["source_labse"] = FENG
cov.to_csv(OUT_PATH, index=False)
check = pd.read_csv(OUT_PATH)
assert check.shape == cov.shape and check["lang"].tolist() == cov["lang"].tolist(), "the saved file does not read back"
print(f"Saved {OUT_PATH.relative_to(ROOT.parent)}: {check.shape[0]} rows, {check.shape[1]} columns")
print(cov[["lang", "afroxlmr_class", "labse_class", "note"]].to_string(index=False))
print("\nAfroXLMR classes:", cov["afroxlmr_class"].value_counts().to_dict(), "; LaBSE:", cov["labse_class"].value_counts().to_dict())

Saved final_project/data/coverage.csv: 12 rows, 10 columns
lang afroxlmr_class labse_class                                                                                 note
 kin        adapted      listed                                                                                     
 swa        adapted      listed                                                                                     
 tso        neither  not_listed                                                                                     
 arq        adapted      listed covered as Standard Arabic (ar) in all three lists; Algerian Arabic is named in none
 por      xlmr_only      listed                     covered as Portuguese (pt); the tweets are Mozambican Portuguese
 twi        neither  not_listed                                                                                     
 pcm        adapted  not_listed                                          Naija (pcm) in the AfroXLMR adaptation only
 ary 

**Note on Cell 1.4.** No computation here: the twelve coverage classes were read on 11 October from Table 9 of Alabi et al. 2022 (the twenty languages of the adaptive fine-tuning that made AfroXLMR), Table 6 of Conneau et al. 2020 (the 100 CC-100 languages XLM-R was pretrained on) and Table 10 of Feng et al. 2022 (the 109 languages of LaBSE), and written to data/coverage.csv with the three sources. Nine languages are in the AfroXLMR adaptation (amh, hau, ibo, kin, pcm, swa, yor, and arq and ary through Standard Arabic), one is in XLM-R's pretraining data but not in the adaptation (por, through Portuguese), and two are in neither (tso, twi). LaBSE lists nine of the twelve and lacks tso, twi and pcm. For arq, ary and por the lists name the standard language, not the variety in the tweets; the note column says so (decision 4), the figure marks them like the others, and the Discussion carries the caveat. Of the two languages no model saw, tso has the smallest training set (804 tweets) and twi the largest share of duplicated training texts, two things the Discussion has to keep apart from the coverage effect.